In [2]:
import gc
import sys
import os
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(project_root)
from datetime import datetime

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import HashingVectorizer, TfidfVectorizer
import joblib
from tqdm import tqdm
from concurrent.futures import as_completed
from common.const import (
    SCHEMA,
    TABLE_RAW_REVIEWS,
    TABLE_FEATURED_USER_VECTORS,
    TABLE_USER_AGGREGATED_STATE,
    TABLE_MATERIALIZED_VIEW_USER,
    TABLE_USER_ITEM_STATE,          
    TABLE_USER_ITEM_STATE_SNAPSHOT,
    TABLE_VECTOR_PRODUCTS,
    TABLE_FEATURED_DENSE_VECTORS,   
    CLICKHOUSE_USERNAME
)
from common.helper import connect_clickhouse, clickhouse_extract_data_as_df
from concurrent.futures import ThreadPoolExecutor
import threading

# =======================
# 1. ClickHouse DDL / MV
# =======================

DDL_USER_AGG_STATE = f"""
CREATE TABLE IF NOT EXISTS {SCHEMA}.{TABLE_USER_AGGREGATED_STATE}
(
    user_id String,

    rating_sum AggregateFunction(sum, Float32),
    rating_count AggregateFunction(count, Float32),
    rating_var AggregateFunction(varSamp, Float32),

    helpful_sum AggregateFunction(sum, UInt32),

    text_array AggregateFunction(groupArray, String),
    category_top AggregateFunction(topK(1), String)
)
ENGINE = AggregatingMergeTree()
ORDER BY user_id;
"""

DDL_MV_USER = f"""
CREATE MATERIALIZED VIEW IF NOT EXISTS {SCHEMA}.{TABLE_MATERIALIZED_VIEW_USER}
TO {SCHEMA}.{TABLE_USER_AGGREGATED_STATE}
AS
SELECT
    user_id,
    sumState(toFloat32(rating)) AS rating_sum,
    countState(rating) AS rating_count,
    varSampState(toFloat32(rating)) AS rating_var,
    sumState(toUInt32(helpful_vote)) AS helpful_sum,
    groupArrayState(concat(
        coalesce(title, ''),
        ' ',
        coalesce(text, ''),
        ' ',
        coalesce(product_category, '')
    )) AS text_array,
    topKState(1)(product_category) AS category_top
FROM {SCHEMA}.{TABLE_RAW_REVIEWS}
GROUP BY user_id;
"""

QUERY_GET_USER_AGG = f"""
SELECT
    user_id,
    sumMerge(rating_sum) / countMerge(rating_count) AS avg_rating,
    varSampMerge(rating_var) AS variance_rating,
    sumMerge(rating_count) AS total_reviews,
    sumMerge(helpful_sum) AS total_helpful_votes,
    arrayStringConcat(groupArrayMerge(text_array), ' ') AS all_reviews_text,
    topKMerge(category_top)[1] AS top_category
FROM {SCHEMA}.{TABLE_USER_AGGREGATED_STATE}
GROUP BY user_id
"""

#Partial state for user–item matrix + static + dynamic (EWMA)
DDL_USER_ITEM_STATE = f"""
CREATE TABLE IF NOT EXISTS {SCHEMA}.{TABLE_USER_ITEM_STATE}
(
    user_id String,

    sum_weighted_emb Array(Float32),        -- Σ w * e
    sum_weight Float32,                     -- Σ w

    sum_emb Array(Float32),                 -- Σ e  (cho static avg)
    count_emb UInt32,                       -- count

    dyn_ewma_emb Array(Float32),            -- EWMA embedding
    dyn_ewma_weight Float32                 -- EWMA tổng trọng số
)
ENGINE = MergeTree()
ORDER BY user_id;
"""

# Feature engineered user vector table
DDL_FEATURED_USER_VECTORS = f"""
CREATE TABLE IF NOT EXISTS {SCHEMA}.{TABLE_FEATURED_USER_VECTORS} (
    id UInt64,
    user_id String,
    avg_rating Float32,
    variance_rating Float32,
    total_ratings UInt32,
    total_reviews UInt32,
    all_reviews_text String,
    top_category String,
    total_helpful_votes UInt32,
    user_text_embedding Array(Float32),
    user_tf_idf_vector Array(Float32),
    user_hashed_vector Array(Float32),
    user_item_matrix_vector Array(Float32),
    static_vector Array(Float32),
    dynamic_vector Array(Float32),
    event_time DateTime DEFAULT now()
) ENGINE = ReplacingMergeTree(event_time)
ORDER BY user_id
PRIMARY KEY user_id;
"""

def drop_clickhouse_tables():
    client = connect_clickhouse()
    # client.command(f"DROP TABLE IF EXISTS {SCHEMA}.{TABLE_USER_AGGREGATED_STATE}")
    # client.command(f"DROP TABLE IF EXISTS {SCHEMA}.{TABLE_USER_ITEM_STATE}")
    # client.command(f"DROP TABLE IF EXISTS {SCHEMA}.{TABLE_FEATURED_USER_VECTORS}")
    # client.command(f"DROP TABLE IF EXISTS {SCHEMA}.{TABLE_MATERIALIZED_VIEW_USER}")
    client.close()

def init_clickhouse_schema():
    client = connect_clickhouse()
    client.command(DDL_USER_AGG_STATE)
    client.command(DDL_MV_USER)
    client.command(DDL_USER_ITEM_STATE)
    client.command(DDL_FEATURED_USER_VECTORS)
    client.close()

def backfill_user_aggregated_state_timestamp():
    client = connect_clickhouse()
    
    # Tìm timestamp range
    ts_stats = client.query("""
        SELECT min(timestamp), max(timestamp) FROM amazon.raw_reviews
    """).result_rows[0]
    min_ts, max_ts = ts_stats
    print(f"Timestamp range: {min_ts} → {max_ts}")
    chunk_days = 10  # 10 days/chunk
    day_ms = 86400000
    current_ts = min_ts
    
    while current_ts < max_ts:
        next_ts = min(current_ts + chunk_days * day_ms, max_ts)
        print(f"Chunk ts={current_ts} → {next_ts}")
        
        q = f"""
        INSERT INTO {SCHEMA}.{TABLE_USER_AGGREGATED_STATE}
        SELECT
            user_id, 
            sumState(toFloat32(rating)) AS rating_sum,
            countState(rating) AS rating_count,
            varSampState(toFloat32(rating)) AS rating_var,
            sumState(toUInt32(helpful_vote)) AS helpful_sum,
            groupArrayState(concat(ifNull(title,''),' ',ifNull(text,''),' ',ifNull(product_category,''))) AS text_array,
            topKState(1)(product_category) AS category_top
        FROM {SCHEMA}.{TABLE_RAW_REVIEWS}
        WHERE timestamp >= {current_ts} AND timestamp < {next_ts}
        GROUP BY user_id
        """
        client.command(q)
        current_ts = next_ts
    
    client.close()
    print("COMPLETE!")


# Run: backfill_user_aggregated_state_chunked()

# ===========================
# 2. Signal text
# ===========================

class UserTextVectorizer:
    def __init__(
        self,
        sbert_model_name: str = "sentence-transformers/all-MiniLM-L6-v2",
        tfidf_path: str | None = None,
    ):
        self.model = SentenceTransformer(sbert_model_name)

        if not tfidf_path or not os.path.exists(tfidf_path):
            raise RuntimeError(
                f"TF-IDF model not found at {tfidf_path}. "
                f"Train it offline on 10k-100k reviews and save with joblib."
            )
        self.tfidf_vectorizer: TfidfVectorizer = joblib.load(tfidf_path)

        self.hashing_vec = HashingVectorizer(
            n_features=1024,
            stop_words="english",
            alternate_sign=False,
            norm="l2",
        )

    def vectorize_user_df(self, user_df: pd.DataFrame) -> pd.DataFrame:
        all_texts = user_df["all_reviews_text"].fillna("").tolist()

        all_emb = self.model.encode(
            all_texts, batch_size=1024, normalize_embeddings=True
        )

        user_df["user_text_embedding"] = [
            v.astype("float32").tolist() for v in all_emb
        ]

        # TF‑IDF + hashing vectorizer
        tfidf_matrix = self.tfidf_vectorizer.transform(all_texts)
        user_df["user_tf_idf_vector"] = [row.tolist() for row in tfidf_matrix.toarray()]

        hashed_matrix = self.hashing_vec.transform(all_texts)
        user_df["user_hashed_vector"] = [row.tolist() for row in hashed_matrix.toarray()]

        return user_df

# ===================================
# 3. User–item state for batch + EWMA
# ===================================

def query_interactions_batch(limit: int, offset: int) -> pd.DataFrame:
    """
    Batch of interaction from raw_reviews:
    need user_id, parent_asin, rating, timestamp.
    """
    q = f"""
    SELECT
        user_id,
        parent_asin AS item_id,
        rating,
        timestamp
    FROM {SCHEMA}.{TABLE_RAW_REVIEWS}
    ORDER BY timestamp
    LIMIT {limit} OFFSET {offset}
    """
    return clickhouse_extract_data_as_df(q)

def query_item_embeddings(item_ids: list[str]) -> pd.DataFrame:
    if not item_ids:
        return pd.DataFrame(columns=["item_id", "emb"])
    unique_ids = list(set(item_ids))
    id_list = ",".join(f"'{i}'" for i in unique_ids)
    q = f"""
    SELECT parent_asin AS item_id, mixed_vector AS emb
    FROM {SCHEMA}.{TABLE_FEATURED_DENSE_VECTORS}
    WHERE parent_asin IN ({id_list})
    """
    return clickhouse_extract_data_as_df(q)

def query_item_embeddings_faster(item_ids: list[str], max_ids_per_query: int = 10000) -> pd.DataFrame:
    """Chunk IN clause to avoid max_query_size"""
    client = connect_clickhouse()
    chunks = [item_ids[i:i+max_ids_per_query] for i in range(0, len(item_ids), max_ids_per_query)]
    dfs = []
    
    for chunk in chunks:
        id_list = ",".join(f"'{x}'" for x in chunk)
        q = f"""
        SELECT parent_asin AS item_id, mixed_vector AS emb
        FROM {SCHEMA}.{TABLE_FEATURED_DENSE_VECTORS}
        WHERE parent_asin IN ({id_list})
        """
        dfs.append(client.query_df(q))
    
    client.close()
    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame(columns=['item_id', 'emb'])

def compute_user_item_partial_state(
    interactions_batch: pd.DataFrame,
    ewma_alpha: float = 0.3,
) -> list[list]:
    """
    compute partial state for each user in batch:
    - sum_weighted_emb, sum_weight for user_item_matrix
    - sum_emb, count_emb for static
    - dyn_ewma_emb, dyn_ewma_weight for dynamic (EWMA on batch)
    EWMA = ClickHouse merge (sum + weight) from each batch
    """
    if interactions_batch.empty:
        return []

    # Load item embeddings
    item_ids = interactions_batch["item_id"].astype(str).tolist()
    item_emb_df = query_item_embeddings(item_ids)
    # item_emb_df = query_item_embeddings(item_ids)
    print('done query', end = ' ')
    if item_emb_df.empty:
        return []

    emb_map = {str(r["item_id"]): np.array(r["emb"], dtype=np.float32)
               for _, r in item_emb_df.iterrows()}

    # Join embedding
    rows = []
    for _, row in interactions_batch.iterrows():
        uid = str(row["user_id"])
        iid = str(row["item_id"])
        rating = float(row["rating"])
        ts = row["timestamp"]

        if iid not in emb_map:
            continue
        emb = emb_map[iid]
        rows.append((uid, ts, rating, emb))

    if not rows:
        return []

    # Group by user in this batch
    data_by_user: dict[str, list[tuple]] = {}
    for uid, ts, rating, emb in rows:
        data_by_user.setdefault(uid, []).append((ts, rating, emb))

    partial_records = []

    for uid, triples in data_by_user.items():
        # Sort by timestamp to apply EWMA in correct order
        triples.sort(key=lambda x: x[0])
        dim = len(triples[0][2])

        sum_weighted = np.zeros(dim, dtype=np.float32)
        sum_weight = 0.0

        sum_emb = np.zeros(dim, dtype=np.float32)
        count_emb = 0

        # EWMA in this batch
        dyn_ewma = np.zeros(dim, dtype=np.float32)
        dyn_weight = 0.0  # effective total weight

        for ts, rating, emb in triples:
            w = abs(rating)
            sum_weighted += emb * w
            sum_weight += w

            sum_emb += emb
            count_emb += 1

            # EWMA update: new_weight = alpha * w + (1 - alpha) * prev_weight
            new_dyn_weight = ewma_alpha * w + (1.0 - ewma_alpha) * dyn_weight
            if new_dyn_weight > 0:
                dyn_ewma = (
                    (1.0 - ewma_alpha) * dyn_ewma * dyn_weight + ewma_alpha * w * emb
                ) / new_dyn_weight
            dyn_weight = new_dyn_weight

        partial_records.append(
            [
                uid,
                sum_weighted.astype("float32").tolist(),
                float(sum_weight),
                sum_emb.astype("float32").tolist(),
                int(count_emb),
                dyn_ewma.astype("float32").tolist(),
                float(dyn_weight),
            ]
        )

    return partial_records


def insert_user_item_partial_state(partial_records: list[list]):
    if not partial_records:
        return
    client = connect_clickhouse()
    columns = [
        "user_id", "sum_weighted_emb", "sum_weight",
        "sum_emb", "count_emb", "dyn_ewma_emb", "dyn_ewma_weight",
    ]

    # Force pure Python types — numpy scalars cause \x80\x80 corruption
    sanitized = []
    for rec in partial_records:
        sanitized.append([
            str(rec[0]),                          # user_id: str
            [float(x) for x in rec[1]],           # sum_weighted_emb: list[float]
            float(rec[2]),                         # sum_weight: float
            [float(x) for x in rec[3]],           # sum_emb: list[float]
            int(rec[4]),                           # count_emb: int
            [float(x) for x in rec[5]],           # dyn_ewma_emb: list[float]
            float(rec[6]),                         # dyn_ewma_weight: float
        ])

    client.insert(
        f"{SCHEMA}.{TABLE_USER_ITEM_STATE}",
        sanitized,
        column_names=columns,
    )
    client.close()
# Query full user behavioral vectors (after ClickHouse merge batch)
QUERY_USER_BEHAVIORAL = f"""
SELECT
    user_id,
    arrayReduce('sum', groupArray(sum_weighted_emb)) AS total_sum_weighted_emb,
    sum(sum_weight) AS total_sum_weight,
    arrayReduce('sum', groupArray(sum_emb)) AS total_sum_emb,
    sum(count_emb) AS total_count_emb,
    arrayReduce('sum', groupArray(dyn_ewma_emb)) AS total_dyn_ewma_emb,
    sum(dyn_ewma_weight) AS total_dyn_ewma_weight
FROM {SCHEMA}.{TABLE_USER_ITEM_STATE}
GROUP BY user_id
"""


# def query_user_behavioral_batch(limit: int, offset: int) -> pd.DataFrame:
#     q = f"""
#     SELECT *
#     FROM ({QUERY_USER_BEHAVIORAL})
#     ORDER BY user_id
#     LIMIT {limit} OFFSET {offset}
#     """
#     return clickhouse_extract_data_as_df(q)

def query_user_behavioral_for_users(user_ids: list[str]) -> pd.DataFrame:
    if not user_ids:
        return pd.DataFrame(columns=[
            "user_id",
            "total_sum_weighted_emb",
            "total_sum_weight",
            "total_sum_emb",
            "total_count_emb",
            "total_dyn_ewma_emb",
            "total_dyn_ewma_weight",
        ])
    id_list = ",".join(f"'{u}'" for u in set(user_ids))
    q = f"""
    SELECT *
    FROM ({QUERY_USER_BEHAVIORAL})
    WHERE user_id IN ({id_list})
    """
    return clickhouse_extract_data_as_df(q)

def compute_final_behavioral_vectors(df: pd.DataFrame) -> pd.DataFrame:
    """
    (sum* / count / weight):
    - user_item_matrix_vector = total_sum_weighted_emb / total_sum_weight
    - static_vector = total_sum_emb / total_count_emb
    - dynamic_vector = total_dyn_ewma_emb  (EWMA)
    """
    def safe_div_array(arr, scalar):
        arr = np.array(arr, dtype=np.float32)
        if scalar <= 0:
            return np.zeros_like(arr, dtype=np.float32).tolist()
        return (arr / scalar).astype("float32").tolist()

    user_item_vecs = []
    static_vecs = []
    dynamic_vecs = []

    for _, row in df.iterrows():
        uim = safe_div_array(row["total_sum_weighted_emb"], row["total_sum_weight"])
        stat = safe_div_array(row["total_sum_emb"], row["total_count_emb"])
        #approximate dynamic vector = EWMA embedding
        dyn = row["total_dyn_ewma_emb"] 
        user_item_vecs.append(uim)
        static_vecs.append(stat)
        dynamic_vecs.append(dyn)

    df["user_item_matrix_vector"] = user_item_vecs
    df["static_vector"] = static_vecs
    df["dynamic_vector"] = dynamic_vecs
    return df


def query_user_features_batch(limit: int, offset: int) -> pd.DataFrame:
    q = f"""
    SELECT *
    FROM ({QUERY_GET_USER_AGG})
    ORDER BY user_id
    LIMIT {limit} OFFSET {offset}
    """
    return clickhouse_extract_data_as_df(q)

def count_rows(table: str) -> int:
    q = f"SELECT count() AS c FROM {SCHEMA}.{table}"
    df = clickhouse_extract_data_as_df(q)
    return int(df["c"][0])

def process_single_inter_batch(i, interaction_batch_size, ewma_alpha):
    """Single batch processor"""
    offset = i * interaction_batch_size
    interactions = query_interactions_batch(interaction_batch_size, offset)
    if interactions.empty:
        return False
    
    partial = compute_user_item_partial_state(interactions, ewma_alpha)
    insert_user_item_partial_state(partial)
    return True

def run_feature_engineering_pipeline(tfidf_path: str, user_batch_size: int = 10000, interaction_batch_size: int = 100000, ewma_alpha: float = 0.3):
    # handle interaction to build user_item_state incremental
    total_interactions = count_rows(TABLE_RAW_REVIEWS)
    client = connect_clickhouse()
    try:
        for i in range(103850000, total_interactions, interaction_batch_size): #build user-item state incremental for each batch of interactions, with EWMA update
            offset = i
            progress_percent = (i / total_interactions) * 100
            print(f"Processing {progress_percent:.3f} offset {offset}...")
            interactions = query_interactions_batch(interaction_batch_size, offset)
            print('done interaction', end=' ')
            if interactions.empty:
                break
            partial = compute_user_item_partial_state(
                interactions_batch=interactions,
                ewma_alpha=ewma_alpha,
            )
            print('done computing', end=' ')
            insert_user_item_partial_state(partial)
            print('done insert')
            del interactions, partial
            gc.collect()

    except:
        client.close()
        raise

    print('Finished processing all interaction batches.')

#query function for better table
def change_backfill_to_mergetree():
    client = connect_clickhouse()
    
    merged_table = f"{SCHEMA}.{TABLE_USER_AGGREGATED_STATE}_merged"
    client.command(f"DROP TABLE IF EXISTS {merged_table}")
    
    client.command(f"""
    CREATE TABLE {merged_table} (
        user_id String,
        avg_rating Float32,
        variance_rating Float32,
        total_reviews UInt32,
        total_helpful_votes UInt32,
        all_reviews_text String
    )
    ENGINE = MergeTree()
    ORDER BY user_id;
    """)
    
    #State suffix bypasses mergeState issues!
    client.command(f"""
    INSERT INTO {merged_table}
    SELECT
        user_id,
        sumMerge(rating_sum) / countMerge(rating_count) AS avg_rating,
        varSampMerge(rating_var) AS variance_rating,
        countMerge(rating_count) AS total_reviews,
        sumMerge(helpful_sum) AS total_helpful_votes,
        arrayStringConcat(groupArrayMerge(text_array), ' ') AS all_reviews_text
    FROM {SCHEMA}.{TABLE_USER_AGGREGATED_STATE}
    GROUP BY user_id;
    """)
    
    client.close()
    print(f"FULL SUCCESS with topK! {merged_table}")

def optimize_featured_dense_vectors():
    client = connect_clickhouse()
    
    #LowCardinality for parent_asin
    client.command(f"""
    ALTER TABLE {SCHEMA}.{TABLE_FEATURED_DENSE_VECTORS} 
    MODIFY COLUMN parent_asin LowCardinality(String)
    """)
    
    #skipping index for parent_asin (optional, extra boost)
    client.command(f"""
    ALTER TABLE {SCHEMA}.{TABLE_FEATURED_DENSE_VECTORS} 
    ADD INDEX idx_asin parent_asin TYPE minmax(1) GRANULARITY 4
    """)
    
    #optimize table to rebuild indexes
    client.command(f"""
    OPTIMIZE TABLE {SCHEMA}.{TABLE_FEATURED_DENSE_VECTORS} 
    FINAL SETTINGS mutations_sync=1
    """)
    
    client.close()

def create_user_item_snapshot_batch(batch_size: int = 500000):
    client = connect_clickhouse()

    snapshot_table = f"{SCHEMA}.{TABLE_USER_ITEM_STATE_SNAPSHOT}"
    state_table = f"{SCHEMA}.{TABLE_USER_ITEM_STATE}"
    DIM = 387

    #table snapshot
    client.command(f"DROP TABLE IF EXISTS {snapshot_table}")
    client.command(f"""
    CREATE TABLE {snapshot_table} (
        user_id String,
        user_item_matrix_vector Array(Float32),
        static_vector Array(Float32),
        dynamic_vector Array(Float32),
        total_sum_weight Float32,
        total_count_emb UInt32,
        total_dyn_ewma_weight Float32
    )
    ENGINE = MergeTree()
    ORDER BY user_id
    """)

    #Lấy danh sách user id
    users = client.query(f"SELECT DISTINCT user_id FROM {state_table}").result_rows
    total_users = len(users)

    print(f"Total users: {total_users}, processing in batches of {batch_size}")

    #Batch insert
    print("Running batch aggregation and insert...")
    for i in range(0, total_users, batch_size):
        batch_users = [u[0] for u in users[i:i+batch_size]]
        user_list_str = ",".join(f"'{u}'" for u in batch_users)

        client.command(f"""
        INSERT INTO {snapshot_table}
        SELECT
            user_id,

            if(total_sum_weight > 0,
            arrayMap(x -> x / total_sum_weight, total_sum_weighted_emb),
            arrayMap(x -> toFloat32(0), range({DIM}))
            ) AS user_item_matrix_vector,

            if(total_count_emb > 0,
            arrayMap(x -> x / total_count_emb, total_sum_emb),
            arrayMap(x -> toFloat32(0), range({DIM}))
            ) AS static_vector,

            if(total_dyn_weight > 0,
            arrayMap(x -> x / total_dyn_weight, total_dyn_emb),
            arrayMap(x -> toFloat32(0), range({DIM}))
            ) AS dynamic_vector,

            total_sum_weight,
            total_count_emb,
            total_dyn_weight

        FROM
        (
            SELECT
                user_id,

                sumForEach(sum_weighted_emb) AS total_sum_weighted_emb,
                sumForEach(sum_emb) AS total_sum_emb,
                sumForEach(dyn_ewma_emb) AS total_dyn_emb,

                sum(sum_weight) AS total_sum_weight,
                sum(count_emb) AS total_count_emb,
                sum(dyn_ewma_weight) AS total_dyn_weight

            FROM {state_table}
            WHERE user_id IN ({user_list_str})
            GROUP BY user_id
        )
        """)

        print(f"Processed batch {i} - {i + len(batch_users)} / {total_users}")

    client.close()
    print("Snapshot created in batches.")

def create_user_item_snapshot():
    client = connect_clickhouse()

    snapshot_table = f"{SCHEMA}.{TABLE_USER_ITEM_STATE_SNAPSHOT}"
    state_table = f"{SCHEMA}.{TABLE_USER_ITEM_STATE}"
    DIM = 387

    print("Creating snapshot table...")

    client.command(f"DROP TABLE IF EXISTS {snapshot_table}")

    client.command(f"""
    CREATE TABLE {snapshot_table} (
        user_id String,
        user_item_matrix_vector Array(Float32),
        static_vector Array(Float32),
        dynamic_vector Array(Float32),
        total_sum_weight Float32,
        total_count_emb UInt32,
        total_dyn_ewma_weight Float32
    )
    ENGINE = MergeTree()
    ORDER BY user_id
    """)

    shards = 32

    for shard in range(shards):

        print(f"Processing shard {shard}/{shards}")

        client.command(f"""
        INSERT INTO {snapshot_table}

        SELECT
            user_id,

            if(total_sum_weight > 0,
                arrayMap(x -> x / total_sum_weight, total_sum_weighted_emb),
                arrayMap(x -> toFloat32(0), range({DIM}))
            ),

            if(total_count_emb > 0,
                arrayMap(x -> x / total_count_emb, total_sum_emb),
                arrayMap(x -> toFloat32(0), range({DIM}))
            ),

            if(total_dyn_weight > 0,
                arrayMap(x -> x / total_dyn_weight, total_dyn_emb),
                arrayMap(x -> toFloat32(0), range({DIM}))
            ),

            total_sum_weight,
            total_count_emb,
            total_dyn_weight

        FROM
        (
            SELECT
                user_id,

                sumForEach(sum_weighted_emb) AS total_sum_weighted_emb,
                sumForEach(sum_emb) AS total_sum_emb,
                sumForEach(dyn_ewma_emb) AS total_dyn_emb,

                sum(sum_weight) AS total_sum_weight,
                sum(count_emb) AS total_count_emb,
                sum(dyn_ewma_weight) AS total_dyn_weight

            FROM {state_table}

            WHERE cityHash64(user_id) % {shards} = {shard}

            GROUP BY user_id
        )
        """)

    client.close()

    print("Snapshot created successfully.")

def migrate_user_item_state_to_mergetree(): #not used anymore, don't use it
    client = connect_clickhouse()

    new_table = f"{TABLE_USER_ITEM_STATE}_v2"
    old_table = f"{SCHEMA}.{TABLE_USER_ITEM_STATE}"
    new_full = f"{SCHEMA}.{new_table}"
    backup_table = f"{SCHEMA}.{TABLE_USER_ITEM_STATE}_backup"

    print("Creating new MergeTree table...")
    client.command(f"""
        CREATE TABLE IF NOT EXISTS {new_full}
        (
            user_id String,
            sum_weighted_emb Array(Float32),
            sum_weight Float32,
            sum_emb Array(Float32),
            count_emb UInt32,
            dyn_ewma_emb Array(Float32),
            dyn_ewma_weight Float32
        )
        ENGINE = MergeTree()
        ORDER BY user_id
    """)

    print("Copying data...")
    client.command(f"""
        INSERT INTO {new_full}
        SELECT *
        FROM {old_table}
    """)

    print("Renaming old table to backup...")
    client.command(f"""
        RENAME TABLE {old_table} TO {backup_table}
    """)

    print("Renaming new table to original name...")
    client.command(f"""
        RENAME TABLE {new_full} TO {old_table}
    """)

    print("Migration completed successfully.")

def merge_snapshot_with_aggregated_state(batch_size: int = 500000):
    client = connect_clickhouse()

    snapshot_table = f"{SCHEMA}.{TABLE_USER_ITEM_STATE_SNAPSHOT}"
    agg_table = f"{SCHEMA}.{TABLE_USER_AGGREGATED_STATE}_with_vectors"
    final_snapshot = f"{SCHEMA}.{TABLE_USER_ITEM_STATE_SNAPSHOT}_FULL"

    # 1) Tạo bảng final snapshot
    client.command(f"DROP TABLE IF EXISTS {final_snapshot}")
    client.command(f"""
    CREATE TABLE {final_snapshot} (
        user_id String,
        avg_rating Float32,
        variance_rating Float32,
        total_reviews Float32,
        total_helpful_votes UInt32,

        user_item_matrix_vector Array(Float32),
        static_vector Array(Float32),
        dynamic_vector Array(Float32),

        user_text_embedding Array(Float32),
        user_tf_idf_vector Array(Float32),
        user_hashed_vector Array(Float32),

        total_sum_weight Float32,
        total_count_emb UInt32,
        total_dyn_ewma_weight Float32
        
    )
    ENGINE = MergeTree()
    ORDER BY user_id
    PRIMARY KEY user_id;
    """)

    # 2) Lấy danh sách user batch
    users = client.query(f"SELECT DISTINCT user_id FROM {snapshot_table}").result_rows
    total_users = len(users)

    for i in range(0, total_users, batch_size):
        batch_users = [u[0] for u in users[i:i+batch_size]]

        # tạo array literal để join
        user_list_str = ",".join(f"'{u}'" for u in batch_users)

        # 3) Insert batch vào final snapshot
        client.command(f"""
        INSERT INTO {final_snapshot}
        SELECT
            s.user_id,
            ifNull(avg_rating, 0) AS avg_rating,
            ifNull(variance_rating, 0) AS variance_rating,
            ifNull(total_reviews, 0) AS total_reviews,
            ifNull(total_helpful_votes, 0) AS total_helpful_votes,

            s.user_item_matrix_vector,
            s.static_vector,
            s.dynamic_vector,

            ifNull(user_text_embedding, arrayMap(x -> toFloat32(0), range(384))) AS user_text_embedding,
            ifNull(user_tf_idf_vector, arrayMap(x -> toFloat32(0), range(384))) AS user_tf_idf_vector,
            ifNull(user_hashed_vector, arrayMap(x -> toFloat32(0), range(384))) AS user_hashed_vector,

            s.total_sum_weight,
            s.total_count_emb,
            s.total_dyn_ewma_weight
            

        FROM {snapshot_table} AS s
        LEFT JOIN {agg_table} AS a USING(user_id)
        WHERE s.user_id IN ({user_list_str})
        """)
        print(f"Processed batch {i} - {i + len(batch_users)} / {total_users}")

    # 3) Xóa snapshot cũ và rename FULL thành snapshot
    client.command(f"DROP TABLE IF EXISTS {snapshot_table}")
    client.command(f"RENAME TABLE {final_snapshot} TO {snapshot_table}")

    client.close()
    print("Merged snapshot created and replaced the old snapshot.")

def drop_user_aggregated_state():
    client = connect_clickhouse()
    client.command(f"DROP TABLE IF EXISTS {SCHEMA}.{TABLE_USER_AGGREGATED_STATE}")
    client.close()

def add_vector_to_aggregated_state(batch_size: int = 20000):

    client = connect_clickhouse()

    vectorizer = UserTextVectorizer(
        tfidf_path=TFIDF_PATH
    )

    source_table = f"{SCHEMA}.{TABLE_USER_AGGREGATED_STATE}_merged"
    target_table = f"{SCHEMA}.{TABLE_USER_AGGREGATED_STATE}_with_vectors"

    BATCH_SIZE = batch_size

    print("Creating target table...")

    client.command(f"""
    CREATE TABLE IF NOT EXISTS {target_table}
    (
        user_id String,
        avg_rating Float32,
        variance_rating Float32,
        total_reviews UInt32,
        total_helpful_votes UInt32,

        user_text_embedding Array(Float32),
        user_tf_idf_vector Array(Float32),
        user_hashed_vector Array(Float32)
    )
    ENGINE = MergeTree()
    ORDER BY user_id
    """)

    result = client.query(f"""
    SELECT max(user_id)
    FROM {target_table}
    """)

    last_user = result.result_rows[0][0] or ""
    print("Start vectorization pipeline")

    while True:
        try:
            query = f"""
            SELECT
                user_id,
                avg_rating,
                variance_rating,
                total_reviews,
                total_helpful_votes,
                all_reviews_text
            FROM {source_table}
            WHERE user_id > '{last_user}'
            ORDER BY user_id
            LIMIT {BATCH_SIZE}
            """

            df = client.query_df(query)

            if df.empty:
                break

            # vectorize
            df = vectorizer.vectorize_user_df(df)

            # drop raw text
            df = df.drop(columns=["all_reviews_text"])

            # insert
            client.insert_df(target_table, df, settings={"max_insert_block_size": 20000})


            last_user = df["user_id"].iloc[-1]
            print(f"Inserted {len(df)} users. Last user: {last_user}")
        except:
            continue

    client.close()

    print("Vectorization finished.")

def cleanup_tables():
    client = connect_clickhouse()
    client.command(f"DROP TABLE IF EXISTS {SCHEMA}.{TABLE_USER_AGGREGATED_STATE}")
    client.command(f"DROP TABLE IF EXISTS {SCHEMA}.{TABLE_USER_AGGREGATED_STATE}_with_vectors")
    client.command(f"DROP TABLE IF EXISTS {SCHEMA}.{TABLE_USER_ITEM_STATE}")
    client.command(f"DROP TABLE IF EXISTS {SCHEMA}.{TABLE_FEATURED_USER_VECTORS}")
    # client.command(f"DROP TABLE IF EXISTS {SCHEMA}.{TABLE_FEATURED_USER_VECTORS}")
    client.command(f"DROP TABLE IF EXISTS {SCHEMA}.{TABLE_MATERIALIZED_VIEW_USER}")
    client.command(f"RENAME TABLE {SCHEMA}.{TABLE_USER_ITEM_STATE_SNAPSHOT} TO {SCHEMA}.{TABLE_FEATURED_USER_VECTORS}")
    client.close()
    print("All tables dropped.")

def build_hnsw_indexes(memory_limit=8_000_000_000):
    """
    Build HNSW indexes sequentially to avoid high RAM usage
    """

    client = connect_clickhouse()
    indexes = [
        ("hnsw_user_item_matrix_vector", "user_item_matrix_vector", 387),
        ("hnsw_static_vector", "static_vector", 387),
        ("hnsw_dynamic_vector", "dynamic_vector", 387),
        ("hnsw_user_text_embedding", "user_text_embedding", 384),
    ]

    # limit RAM during index build
    client.command(f"SET max_memory_usage = {memory_limit}")
    client.command("SET allow_experimental_vector_similarity_index = 1")

    for index_name, column, dim in indexes:
        print(f"\nCreating index: {index_name}")

        add_index_query = f"""
        ALTER TABLE {SCHEMA}.{TABLE_FEATURED_USER_VECTORS}
        ADD INDEX IF NOT EXISTS {index_name}
        {column}
        TYPE vector_similarity(
            'hnsw',
            'cosineDistance',
            {dim},
            'bf16',
            16,
            128
        )
        GRANULARITY 1
        """

        client.command(add_index_query)

        print(f"Materializing index: {index_name}")

        materialize_query = f"""
        ALTER TABLE {SCHEMA}.{TABLE_FEATURED_USER_VECTORS}
        MATERIALIZE INDEX IF EXISTS {index_name}
        """

        client.command(materialize_query)

        print(f"Finished {index_name}")
    client.close()
    print("\nAll HNSW indexes built successfully.")

def build_product_vector_indexes(memory_limit=8_000_000_000):
    client = connect_clickhouse()

    indexes = [
        ("hnsw_vector_high_low_signal", "vector_high_low_signal", 384),
        ("hnsw_mixed_vector", "mixed_vector", 387),
    ]

    client.command(f"SET max_memory_usage = {memory_limit}")
    client.command("SET allow_experimental_vector_similarity_index = 1")

    for index_name, column, dim in indexes:

        print(f"\nCreating index: {index_name}")

        add_index_query = f"""
        ALTER TABLE {SCHEMA}.{TABLE_FEATURED_DENSE_VECTORS}
        ADD INDEX IF NOT EXISTS {index_name}
        {column}
        TYPE vector_similarity(
            'hnsw',
            'cosineDistance',
            {dim},
            'bf16',
            32,
            200
        )
        GRANULARITY 1
        """

        client.command(add_index_query)

        print(f"Materializing index: {index_name}")

        materialize_query = f"""
        ALTER TABLE {SCHEMA}.{TABLE_FEATURED_DENSE_VECTORS}
        MATERIALIZE INDEX IF EXISTS {index_name}
        """

        client.command(materialize_query)

        print(f"Finished {index_name}")
    client.close()
    print("\nAll product vector indexes built successfully.")
# ===========================
# 1) init schema
# drop_clickhouse_tables(); print('finish drop tables')
### backfill_user_aggregated_state_timestamp() #only run once to backfill
### optimize_featured_dense_vectors() run already, only run once
### change_backfill_to_mergetree()
client = connect_clickhouse()
# init_clickhouse_schema()
query = f"""ALTER USER {CLICKHOUSE_USERNAME} SETTINGS 
    max_query_size = 1000000000,
    max_ast_elements = 1000000;"""
client.command(query)
build_product_vector_indexes()
# build_hnsw_indexes()
### cleanup_tables()
#FOR SNAPSHOT
# create_user_item_snapshot_batch()
TFIDF_PATH = "tfidf_vectorizer.pkl"

add_vector_to_aggregated_state(batch_size = 20000)

# create_user_item_snapshot()
# merge_snapshot_with_aggregated_state()
print('finish init schema')
# run_feature_engineering_pipeline(
#     tfidf_path=TFIDF_PATH,
#     user_batch_size=200000,
#     interaction_batch_size=250000,
#     ewma_alpha=0.3
#     )

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Creating target table...
Start vectorization pipeline
Inserted 20000 users. Last user: AE4KNWOH5Y3RKFNN7GTXPQRJPVEQ
Inserted 20000 users. Last user: AE4NDE3T6TESH5GHB664JACSP6VA
Inserted 20000 users. Last user: AE4PZ5AN5UMWCI5JHUB74PDLG4LA
Inserted 20000 users. Last user: AE4SNN52ZNEJUKHHVWROWSV5CQXQ
Inserted 20000 users. Last user: AE4VE73QMVTYCYWTOSW7F6WOBHZQ
Inserted 20000 users. Last user: AE4XZNFXGALMAM3V7YSOUSWPWTOQ
Inserted 20000 users. Last user: AE52P6WLO6X2UTFV54YYQHQO5UTA
